### Week 5 Day 4

AutoGen Core - Distributed

I'm only going to give a Teaser of this!!

Partly because I'm unsure how relevant it is to you. If you'd like me to add more content for this, please do let me know..

In [38]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from IPython.display import display, Markdown
from dotenv import load_dotenv
import os

load_dotenv(override=True)

ALL_IN_ONE_WORKER = True

# Helper function to create Groq client
def create_groq_client(model_name: str):
    return OpenAIChatCompletionClient(
        model=model_name,
        base_url="https://api.groq.com/openai/v1",
        api_key=os.getenv("GROQ_API_KEY"),
        model_info={
            "vision": False,
            "function_calling": True,
            "json_output": True,
            "structured_output": True,
            "family": "unknown",
        },

    )

### Start with our Message class

In [39]:

@dataclass
class Message:
    content: str

### And now - a host for our distributed runtime

In [43]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

### Let's reintroduce a tool

In [44]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [45]:
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

### And make some Agents

In [46]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        # Check for API key
        if not os.getenv("GROQ_API_KEY"):
            raise ValueError("GROQ_API_KEY not found in environment variables")
        
        # Use helper function for openai/gpt-oss-120b with tools
        model_client = create_groq_client("openai/gpt-oss-120b")
        self._delegate = AssistantAgent(
            name, 
            model_client=model_client, 
            tools=[autogen_serper], 
            reflect_on_tool_use=False
        )

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        # Check for API key
        if not os.getenv("GROQ_API_KEY"):
            raise ValueError("GROQ_API_KEY not found in environment variables")
        
        # Use helper function for llama-3.3-70b-versatile with tools
        model_client = create_groq_client("openai/gpt-oss-120b")
        self._delegate = AssistantAgent(
            name, 
            model_client=model_client, 
            tools=[autogen_serper], 
            reflect_on_tool_use=False
        )

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        # Check for API key
        if not os.getenv("GROQ_API_KEY"):
            raise ValueError("GROQ_API_KEY not found in environment variables")
        
        # Use helper function for moonshotai/kimi-k2-instruct-0905 as judge
        model_client = create_groq_client("moonshotai/kimi-k2-instruct-0905")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)

In [47]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:

    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")




In [48]:
response = await worker.send_message(Message(content="Go!"), agent_id)

In [51]:
print(response)

Message(content="## Pros of AutoGen:\nDesigned for seamless collaboration between agents, AutoGen empowers developers to create robust, scalable and intelligent multi-agent systems. Missing: pros benefits. Key Benefits for Enterprise AI Development · Reduced Coordination Complexity: Natural-language handoffs eliminate custom inter-agent protocols, ... Missing: pros | Show results with:pros. AutoGen is a vehicle for AI Frontiers to turn state of the art research into agentic capabilities and enable the development of AI applications ... Missing: pros | Show results with:pros. Agentic Frameworks (AutoGen, CrewAI) · Advanced AI Capabilities: Great for complex automation and dynamic environments. · High Customization: ... AutoGen's strengths lie in its powerful conversation-driven orchestration, extensibility, and the rich feature set for building autonomous (or ... AutoGen improves efficiency, scalability, and problem-solving depth by fostering collaboration between agents in AI-powered s

In [52]:
display(Markdown(response.content))


## Pros of AutoGen:
Designed for seamless collaboration between agents, AutoGen empowers developers to create robust, scalable and intelligent multi-agent systems. Missing: pros benefits. Key Benefits for Enterprise AI Development · Reduced Coordination Complexity: Natural-language handoffs eliminate custom inter-agent protocols, ... Missing: pros | Show results with:pros. AutoGen is a vehicle for AI Frontiers to turn state of the art research into agentic capabilities and enable the development of AI applications ... Missing: pros | Show results with:pros. Agentic Frameworks (AutoGen, CrewAI) · Advanced AI Capabilities: Great for complex automation and dynamic environments. · High Customization: ... AutoGen's strengths lie in its powerful conversation-driven orchestration, extensibility, and the rich feature set for building autonomous (or ... AutoGen improves efficiency, scalability, and problem-solving depth by fostering collaboration between agents in AI-powered solutions. Why Multi ... Missing: pros | Show results with:pros. Microsoft's AutoGen frameworks stand out as a powerful tool for creating and managing multi-agent conversations. AutoGen simplifies the process ... AutoGen Advantages. Open Source: Free to use with complete code transparency. Customization: Full control over agent architecture and behavior. Gagan Bansal, Senior Researcher, Microsoft Research AI Frontiers introduces a transformative update to the AutoGen framework that builds on user ...

## Cons of AutoGen:
autogens main problems are that its documentation is quite hard to read, with not enough examples, and there are somethings that flat out don't ... ... AI? Let's chat: https://www.brainqub3.com/book-online Register your interest ... AutoGen workflow: 11:20 Other AutoGen limitations: 33:10. Missing: framework | Show results with:framework. This is my opinionated, experience-based comparison of the three most talked-about frameworks in the agentic AI space today: CrewAI, LangGraph, AutoGen. The problem with that is it leads to over-engineered systems. Frameworks like CrewAI and Autogen provide a fully autonomous agentic system. That ... In this Crewai vs Autogen article, we explain the difference between the two and conclude which one is the best to build AI agents and ... The real problem with Agentic frameworks like CrewAI or Autogen is AGENTS! The idea of agents is quite a seductive one! This report provides a detailed comparison of SmythOS and AutoGen, examining their key differences and benefits across various dimensions. The main drawback is complexity – defining and debugging a graph requires a moderate learning curve. ... code AutoGen Studio is in preview) ... AutoGen: Experimental Framework Limitations. Research-Focused Design: AutoGen is best suited for research teams and experimental AI projects.



## Decision:

DECISION: Do NOT use AutoGen for this project.

RATIONALE: While AutoGen offers powerful multi-agent orchestration, the research consensus is that it is still an experimental, research-oriented framework. The cons highlight hard-to-read documentation, sparse examples, and a tendency to encourage over-engineered solutions. These factors outweigh the benefits for a production deliverable that needs rapid onboarding, maintainability, and clear engineering guardrails.

In [53]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [54]:
await host.stop()